# RgDAAO–ESM-2 robustness study

Select **Runtime → Change runtime type → T4 GPU**, then run in a fresh runtime.
The original seed-42 benchmark remains separate. This notebook runs the repository code.

Default: **position-held-out split, training seeds 42/43/44** (six model runs).
It reserves whole residue positions and keeps that partition fixed across training seeds.
Based on the initial T4 run, budget roughly **3 hours plus setup**; runtime can vary.
Run the random split in a second fresh session, or enable both strategies for about six hours.
No additional performance claim exists until these experiments finish.


In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path
repo = Path('/content/RgDAAO-ESM2')
if repo.exists():
    raise RuntimeError('Use a fresh Colab runtime to avoid overwriting previous experiments.')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/jjimenezgar/RgDAAO-ESM2.git', str(repo)], check=True)
os.chdir(repo)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[ml]'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before starting.'
print('GPU:', torch.cuda.get_device_name(0))
from transformers import EsmForSequenceClassification
for script in ('download_data.py', 'build_dataset.py'):
    subprocess.run([sys.executable, 'scripts/' + script], check=True)

def run_visible(command):
    print('$', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Command failed. Inspect the visible log above.')
run_visible([sys.executable, 'scripts/smoke_test.py'])


## Lock the experiment plan
Edit the settings before running this cell. Use `['random']` in a second session to
measure training variation on the original random split, or `['position', 'random']`
to run both. Three seeds measure training variation; the split seed stays fixed.
Do not choose settings based on new test scores. Preparation audits split structure
but does not fit models or use test labels for model selection.


In [ ]:
STRATEGIES = ['position']
TRAINING_SEEDS = [42, 43, 44]
SPLIT_SEED = 42
OUTPUT = Path('results/robustness')
run_visible([sys.executable, 'scripts/run_robustness.py', '--prepare-only',
             '--output', str(OUTPUT), '--strategies', *STRATEGIES,
             '--seeds', *map(str, TRAINING_SEEDS), '--split-seed', str(SPLIT_SEED)])
print((OUTPUT / 'plan.json').read_text())


## Train, select and evaluate
All planned frozen/fine-tuned models finish training and validation-based selection
before any new test evaluation. The runner then writes per-run figures, `runs.csv`,
`aggregate.csv`, `paired_deltas.csv`, `paired_summary.csv` and `summary.md`.
Rerunning this cell in the same runtime skips completed training/evaluation stages.
It cannot resume an interrupted training job; preserve its files and use a fresh
output directory for a new complete plan. A fresh runtime does not retain local files.


In [ ]:
run_visible([sys.executable, 'scripts/run_robustness.py', '--resume', '--output', str(OUTPUT)])
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'summary.md').read_text()))


## Download the evidence
Run this cell after completion. You can also run it manually after a failure to
preserve partial evidence; a partial archive is not a completed scientific result.
Selected model weights are retained. Intermediate optimizer checkpoints are
excluded from the export to keep its size manageable. The archive includes the
locked plan, full splits, configurations, execution records and predictions.


In [ ]:
import zipfile
from google.colab import files
archive = Path('/content/RgDAAO-ESM2-robustness-' + '-'.join(STRATEGIES) + '.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file() and 'checkpoints' not in path.relative_to(OUTPUT).parts:
            zipped.write(path, path.relative_to(OUTPUT))
print('Saved evidence:', archive)
files.download(str(archive))
